In [ ]:
from bs4 import BeautifulSoup,Tag
import requests
from html_to_markdown import convert_to_markdown

url = "https://sicstus.sics.se/sicstus/docs/latest4/html/sicstus.html/index.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')  # or 'lxml'

contents = soup.select_one('body > div > ul.no-bullet')
print(contents)

In [ ]:
def get_section(contents:Tag,section) -> Tag | None:
    for t in contents.descendants:
        if t.name == 'ul':
            possible_result = get_section(t,section)
            if possible_result:
                return possible_result
        elif t.name == 'li':
            if t.text.strip().split(' ')[0] == section:
                return t
        else:
            continue
    return None

In [ ]:
built_in_predicates = get_section(contents,"11.3")
built_in_predicates
number_builtins = len(list(built_in_predicates.children))
number_builtins

In [ ]:
teste = get_section(contents,"11.3.60")
teste

In [ ]:
def concat_ps(ps:list[Tag])->str:
    p_tags = [convert_to_markdown(p) for p in ps if p.name is not None]
    return "\n".join(p_tags)

class PredicateDocBuilder:
    def __init__(self):
        self.compat = None
        self.signature = None
        self.templates = []
        self.synopsis =None
        self.description = None
        self.arguments = []
        self.examples = None
        self.see_also = None
        self.exception = None
        self.backtracking =  None
        self.comments = None
        self.tips = None
        self.tags = []
        pass
    def handle_synopsis(self,contents):
        name, arity = self.signature
        # TODO if arity is list there will be more templates
        n_templates = 1
        contents = [c for c in contents if c.name is not None]
        if type(arity) is list:
            n_templates = len(arity)
        assert len(contents) > (n_templates), f"In predicate {name}/{arity} the synopsis is probably missing information"
        for i in range(n_templates):
            template_p : Tag= contents[i]
            self.templates.append(template_p.get_text(strip=True))
        self.synopsis = concat_ps(contents[n_templates:])

    def handle_description(self,contents):
        self.description =  concat_ps(contents)
    def handle_comments(self,contents):
        self.comments =  concat_ps(contents)

    def handle_tips(self,contents):
        self.tips =  concat_ps(contents)

    def handle_arguments(self,contents):
        arguments = []
        contents = [c for c in contents if c.name is not None]
        if len(contents) > 0 and contents[0].name == 'dl':
            contents = list(contents[0].children)
        contents = [c for c in contents if c.name is not None]
        if len(contents) == 1 and contents[0].name == 'dd':
            dl = contents[0].dl
            if dl is None:
                assert len(contents) % 2 == 0, f"Predicate {self.signature} has weird parameters"
            contents = [c for c in dl.children if c.name is not None]
            
                
        contents = [c for c in contents if c.name is not None]
        assert len(contents) % 2 == 0, f"Predicate {self.signature} has weird parameters"
        for i in range(0,len(contents),2):
            term = contents[i].get_text(strip=True)
            definition = convert_to_markdown(contents[i+1])
            arguments.append((term,definition))

        self.arguments = arguments
    def handle_examples(self,contents):
        return 
    def handle_see_also(self,contents):
        self.see_also = concat_ps(contents)
    def handle_exception(self,contents):
        self.exceptions= concat_ps(contents)
    def handle_backtracking(self,contents):
        self.backtracking= concat_ps(contents)
    
    def handle_keywords(self,predicate_name):
        tags = predicate_name.select('i')
        for t in tags:
            self.tags.append(t.text)


    def handle_predicate_name(self,predicate_name):
        signature = predicate_name.text
        parts = signature.split('/')
        assert len(parts) == 2

        arity = parts[1]
        if parts[1][0] == '[':
            numbers = parts[1][1:-1].split(',')
            if all(n.isdigit() for n in numbers):
                arity = [int(n) for n in numbers]
            else:
                print(arity)
        elif parts[1].isdigit():
            arity = int(parts[1])
        self.signature = (parts[0],arity)
    
    def to_pldoc(self):
        r = ""
        def append_line(s,new_line = False):
            res = ""
            for i, line in enumerate(s.split('\n')):
                line= line.strip()
                if len(line) > 0:
                    begin = "%     " if i != 0 or new_line else ""
                    res +=  begin +  line  + "\n"
            return res
        for t in self.templates:
            r +=  "%!    " + t  + "\n"
        r += "%\n"
        r += append_line(self.synopsis,new_line = True)
        r += "%\n" if self.synopsis and len(self.arguments)>0 else ""
        max_len = max([len(argument) for (argument,_) in self.arguments]) if len(self.arguments) > 0 else 0
        for argument, description in self.arguments:
            r +=  f"%     @{argument}"  + (max_len+3 - len(argument))*' ' + append_line(description)
            
        return r

    def build(self,page):
        predicate_name = page.select("h4.subsection")
        assert len(predicate_name) == 1, "Found a page without predicate name"
        self.handle_predicate_name(predicate_name[0].select("code")[0])
        
        self.handle_keywords(predicate_name[0])

        subsections = page.select("h4.subheading")

        section_functions =  {
            'Synopsis': self.handle_synopsis,
            'Description': self.handle_description,
            'Arguments': self.handle_arguments,
            'Examples': self.handle_examples,
            'See Also': self.handle_see_also,
            'Exceptions': self.handle_exception,
            'Comments': self.handle_comments,
            'Tips': self.handle_tips,
            'Backtracking': self.handle_backtracking,
        }

        for section in subsections:
            name = section.text
            if name in section_functions:
                section_functions[name](self.section_contents(section))
            else:
                print("No registered function to handle:",name)
                raise TypeError

    def section_contents(self,tag:Tag)-> list[Tag]:
        result = []
        current_symbol = tag.next_sibling
        while current_symbol is not None and current_symbol.name != "h4":
            result.append(current_symbol)
            current_symbol = current_symbol.next_sibling
        return result
    def print(self):
        if self.compat != None:
            print(f"compat:{self.compat}")
        if self.signature != None:
            print(f"name:{self.signature}")
        if self.synopsis !=None:
            print(f"synopsis:{self.synopsis}")
        if self.description != None:
            print(f"description:{self.description}")
        if self.arguments != None:
            print(f"arguments:{self.arguments}")
        if self.examples != None:
            print(f"examples:{self.examples}")
        if self.see_also != None:
            print(f"see_also:{self.see_also}")
        if self.exception != None:
            print(f"exception:{self.exception}")
        if self.backtracking !=  None:
            print(f"backtracking:{self.backtracking}")
        if self.tags != []:
            print(f"tags:{self.tags}")
    


        

In [ ]:
import os
def get_predicate_page(predicate_li: Tag):
    name = predicate_li.code.text.split('/')[0]
    
    file_path = f'builtins/{name}.html'
    if os.path.exists(file_path):
        result = ""
        with open(file_path,'r') as f:
            result = f.read()
        return result
    
    print(f"Predicate {name}, not stored")
    member_page_link = predicate_li.select('a')[0].get('href')
    base_link = "https://sicstus.sics.se/sicstus/docs/latest4/html/sicstus.html/"
    response = requests.get(base_link + member_page_link)
    if response.status_code != 200:
        raise Exception(response.status_code)
    with open(file_path) as f:
        f.write(response.text)
    return response.text

def visit_predicate(predicate_li:Tag):

    page = get_predicate_page(predicate_li)
    soup = BeautifulSoup(page, 'html.parser')  # or 'lxml'
    predicate_builder = PredicateDocBuilder()
    predicate_builder.build(soup)
    return predicate_builder

builtin_predicates = built_in_predicates.select("ul")[0].children

result = []
for predicate in builtin_predicates:
    if predicate.name is None:
        continue
    elif predicate.name == 'li':
        p = visit_predicate(predicate)
        result.append(p)
    elif predicate.name == 'ul':
        print(predicate)
    else:
        print(predicate)


In [ ]:
with open('builtins.pl','w') as f:
    for predicate in result:
        f.write(predicate.to_pldoc())
        f.write('\n')

In [ ]:
print(result[0].to_pldoc())